# Sales & Demand Forecasting
## Notebook 3: Model Building

### Objective
Build forecasting models using the prepared weekly sales data and generate predictions for the test period.

### Scope
- Load the prepared weekly sales dataset.
- Split the data chronologically into training and testing sets.
- Build a Naive Baseline model.
- Build a Linear Regression model.
- Save predictions for model evaluation.

Model performance will be evaluated in Notebook 4.

## 1. Load the Prepared Dataset

Load the weekly sales dataset created in Notebook 2. Inspect its structure and confirm that the features required for model building are available.

In [19]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_DIR = Path("..")
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

# Load prepared weekly sales data
weekly_sales = pd.read_csv(
    PROCESSED_DATA_DIR / "weekly_sales_features.csv",
    parse_dates=["date"]
)


print("Dataset shape:", weekly_sales.shape)
print("\nColumns:", weekly_sales.columns.tolist())
print("\nDate range:", weekly_sales["date"].min(),
      "to", weekly_sales["date"].max())

print("\nMissing values:")
print(weekly_sales.isnull().sum())

display(weekly_sales.head())

Dataset shape: (241, 6)

Columns: ['date', 'sales', 'year', 'month', 'week_of_year', 'time_index']

Date range: 2013-01-06 00:00:00 to 2017-08-13 00:00:00

Missing values:
date            0
sales           0
year            0
month           0
week_of_year    0
time_index      0
dtype: int64


,date,sales,year,month,week_of_year,time_index
0,2013-01-06,2.211570e+06,2013,1,1,0
1,2013-01-13,2.373618e+06,2013,1,2,1
2,2013-01-20,2.368007e+06,2013,1,3,2
3,2013-01-27,2.272165e+06,2013,1,4,3
4,2013-02-03,2.476891e+06,2013,2,5,4


## 2. Split the Data into Training and Testing Sets

Use a chronological split, keeping the first 80% of observations for training and the final 20% for testing.

This approach preserves the time order of the data and ensures that the models are tested on later weeks rather than randomly selected observations.

In [20]:
# Sort data chronologically
weekly_sales = (
    weekly_sales
    .sort_values("date")
    .reset_index(drop=True)
)

# Create an 80/20 chronological split
split_idx = int(len(weekly_sales) * 0.8)

train_data = weekly_sales.iloc[:split_idx].copy()
test_data = weekly_sales.iloc[split_idx:].copy()

print("Training data shape:", train_data.shape)
print("Testing data shape:", test_data.shape)

print("\nTraining date range:")
print(train_data["date"].min(), "to", train_data["date"].max())

print("\nTesting date range:")
print(test_data["date"].min(), "to", test_data["date"].max())

Training data shape: (192, 6)
Testing data shape: (49, 6)

Training date range:
2013-01-06 00:00:00 to 2016-09-04 00:00:00

Testing date range:
2016-09-11 00:00:00 to 2017-08-13 00:00:00


## 3. Build the Naive Baseline Model

The Naive Baseline provides a simple reference for forecasting performance. It uses the last observed sales value from the training data as the prediction for every week in the test period.

The predictions from this model will be compared with those from Linear Regression during model evaluation.

In [21]:
# Get the last observed sales value from training data
last_train_sales = train_data["sales"].iloc[-1]

# Predict the same value for every test week
naive_predictions = [last_train_sales] * len(test_data)

print("Last training week's sales:", last_train_sales)
print("Number of predictions:", len(naive_predictions))

Last training week's sales: 5750789.4229352
Number of predictions: 49


## 4. Build the Linear Regression Model

Linear Regression models the relationship between the continuous time index and weekly sales. The time index represents the progression of weeks and helps the model learn the overall long-term sales trend.

Train the model using the training data and generate predictions for the test period.

In [22]:
from sklearn.linear_model import LinearRegression

# Select the input feature and target
X_train = train_data[["time_index"]]
y_train = train_data["sales"]

X_test = test_data[["time_index"]]

# Create and train the model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Generate predictions
linear_predictions = linear_model.predict(X_test)

print("Linear Regression model trained successfully.")
print("Number of predictions:", len(linear_predictions))

Linear Regression model trained successfully.
Number of predictions: 49


## 5. Save Model Predictions

Combine the actual sales values with predictions from both models and save them to a CSV file.

The saved predictions will be used in Notebook 4 to evaluate and compare model performance.

In [23]:
# Create a table of actual sales and model predictions
predictions = test_data[["date", "sales"]].copy()

predictions = predictions.rename(
    columns={"sales": "actual_sales"}
)

predictions["naive_prediction"] = naive_predictions
predictions["linear_regression_prediction"] = linear_predictions

# Create output directory
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save predictions
output_path = OUTPUT_DIR / "model_predictions.csv"
predictions.to_csv(output_path, index=False)

print("Predictions saved to:", output_path)

display(predictions.head())

Predictions saved to: ..\outputs\model_predictions.csv


,date,actual_sales,naive_prediction,linear_regression_prediction
192,2016-09-11,5.492016e+06,5.750789e+06,5.872551e+06
193,2016-09-18,5.084586e+06,5.750789e+06,5.891177e+06
194,2016-09-25,5.010213e+06,5.750789e+06,5.909803e+06
195,2016-10-02,5.539344e+06,5.750789e+06,5.928430e+06
196,2016-10-09,5.706731e+06,5.750789e+06,5.947056e+06


## 6. Model Building Summary

- Loaded the prepared weekly sales dataset containing 241 observations.
- Created a chronological train-test split, with 192 weeks for training and 49 weeks for testing.
- Built a Naive Baseline model using the last training week's sales value.
- Trained a Linear Regression model using the continuous `time_index` feature.
- Generated predictions for both models for all 49 test weeks.
- Saved the actual sales values and model predictions to `outputs/model_predictions.csv`.

### Next Step

Proceed to Notebook 4: Model Evaluation to evaluate both models using MAE, RMSE, and R², and compare their forecasting performance.

No conclusions about which model performs better have been made at this stage.